<a href="https://colab.research.google.com/github/INSVikrant54/Neural-Network/blob/main/NeuralNetwork_(8).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install nnfs

In [ ]:
import numpy as np
import nnfs
from nnfs.datasets import spiral_data

nnfs.init()

class Layer_Dense:
  def __init__(self, n_inputs, n_neurons):
    self.weights = 0.10 * np.random.randn(n_inputs, n_neurons)  #multiplying by 0.10 to get values betwwen -1 and 1
    self.biases = np.zeros((1, n_neurons))    # 1- is the shape of bias and also no need for Transpose as np.np.random.randn handles that
  def forward(self, inputs):
    self.output = np.dot(inputs, self.weights) + self.biases

#Rectified Linear(ReLU) OBJECT
class Activation_ReLU:
  def forward(self, inputs):
    self.output = np.maximum(0, inputs)

class Activation_SoftMax:
  def forward(self, inputs):
    exp_values = np.exp(inputs - np.max(inputs, axis=1, keepdims=True))
    probabilities = exp_values / np.sum(exp_values, axis=1, keepdims=True)
    self.output = probabilities

class Loss:
  def calculate(self, output, y):   #output :output from model , y : is the intended target value
    sample_losses = self.forward(output, y)
    data_loss = np.mean(sample_losses)
    return data_loss

class Loss_CategoricalCrossEntropy(Loss):
  def forward(self, y_pred, y_true):  #y_pred : values from the neural netwoek , y_true : target training value
    samples = len(y_pred)
    y_pred_clipped = np.clip(y_pred, 1e-7, 1-1e-7)  #values clipped to count INFINITY when we get log(0)

    if len(y_true.shape) == 1:    #for scalar inputs
      correct_confidences = y_pred_clipped[range(samples), y_true]

    elif len(y_true.shape) == 2:    #for batch or matrix inputs
      correct_confidences = np.sum(y_pred_clipped * y_true, axis=1)

    negative_log_likelihoods = -np.log(correct_confidences)
    return negative_log_likelihoods


X, y = spiral_data(samples=100, classes=3)

dense1 = Layer_Dense(2 ,3)   #(2- no. of features per input/sample(here spirial dataset has x and y axis), 3- no of neurons we nedd(LITERALLY CAN BE ANY VALUE))
activation1 = Activation_ReLU()

dense2 = Layer_Dense(3, 3)
activation2 = Activation_SoftMax()

dense1.forward(X)
activation1.forward(dense1.output)  #We are inputing the calulatedd valuess in ReLU to make -ve values to 0 and +ve values remains same

dense2.forward(activation1.output)  #passing the ReLU values to SoftMax Fn
activation2.forward(dense2.output)

print(activation2.output[:5])  #shows only 5 batches

loss_function = Loss_CategoricalCrossEntropy()
loss = loss_function.calculate(activation2.output, y)

print("Loss : ", loss)


#(in Next File added)
#

[[0.33333334 0.33333334 0.33333334]
 [0.33331734 0.3333183  0.33336434]
 [0.3332888  0.33329153 0.33341965]
 [0.33325943 0.33326396 0.33347666]
 [0.33323312 0.33323926 0.33352762]]
Loss :  1.098445


Calculating loss for inputs in Batchess

In [ ]:
import numpy as np

softmax_outputs = np.array([[0.7, 0.1, 0.2],
                           [0.1, 0.5, 0.4],
                           [0.02, 0.9, 0.08]])
class_targets = [0, 1, 1]

print(-np.log(softmax_outputs[
    range(len(softmax_outputs)), class_targets
    ]))


[0.35667494 0.69314718 0.10536052]


**In This method we have a problem **

---



*   when we have 0 in softmax fn
then log(0) = INFINTY.
So, we need to avoid that
*   **SO to avoid that we need to clip the values in the range with barely some insignificant amount (something barely to near to close value of 0)
like 1e^-7**

